In [18]:
spark.stop()

In [1]:
from pyspark.sql import SparkSession

# spark = (
#     SparkSession.builder
#     .appName("Przygotowanie")
#     # .master("local[*]")
#     .master("spark://spark-master:7077")
#     .getOrCreate()
# )


spark = (
    SparkSession.builder
    .appName("Przygotowanie")
    .master("spark://spark-master:7077")
    # .master("local[*]")    
    .config("spark.sql.warehouse.dir", "/opt/spark/warehouse/")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 17:36:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.sql("show tables").show()

26/03/20 17:36:41 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/20 17:36:41 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/03/20 17:36:43 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/03/20 17:36:43 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.5
26/03/20 17:36:43 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|uam_categories|      false|
|  default|    uam_offers|      false|
|  default|    uam_orders|      false|
+---------+--------------+-----------+



In [3]:
spark.sql("select * from uam_offers").show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------------------+--------------------+--------------------+-------------------+-------------------+--------+---------+------------+---------------------+----------------------+----------------------+----------------------+-------------+--------------------+---+
|            offer_id|          offer_name|           seller_id|      starting_time|        ending_time|duration|    types|buynow_price|auction_minimal_price|auction_starting_price|stock_initial_quantity|stock_current_quantity|category_leaf|          attributes| pv|
+--------------------+--------------------+--------------------+-------------------+-------------------+--------+---------+------------+---------------------+----------------------+----------------------+----------------------+-------------+--------------------+---+
|004172c9c5049a158...|Części samochodow...|eb4dd0d58f2b23c39...|2017-03-06 14:56:46|               NULL|    NULL|[BUY_NOW]|         170|                 NULL|                  NULL|                  

In [3]:
!ls /home/jovyan/data/

ls: cannot access '/home/jovyan/data/': No such file or directory


In [17]:
spark.sql("select * from `parquet`.`/opt/spark/warehouse/uam_categories.snappy.parquet`").show()

26/03/20 17:32:15 WARN ObjectStore: Failed to get database parquet, returning NoSuchObjectException
26/03/20 17:32:15 WARN TaskSetManager: Lost task 0.0 in stage 2.0 (TID 8) (172.18.0.3 executor 1): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:387)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:443)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:493)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:485)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParall

AnalysisException: [UNSUPPORTED_DATASOURCE_FOR_DIRECT_QUERY] Unsupported data source type for direct query on files: parquet; line 1 pos 14

In [4]:
import shutil
import urllib.request


# catalog = "workspace"
schema = "default"
volume = "uam"

config = [
    {   
        'table': 'uam_categories',
        'url': 'https://drive.google.com/u/1/uc?id=1WLPzOczLl0ATgk2qyRpLUdrG6duDhkL2&export=download',
        'expected_row_cnt': 174
    },
    {
        'table': 'uam_orders',
        'url': 'https://drive.google.com/u/1/uc?id=1zt-YtImHn_48pgNWNQ-xZQSbcoxbrdHe&export=download',
        'expected_row_cnt': 100
    },
    {
        'table': 'uam_offers',
        'url': 'https://drive.google.com/u/1/uc?id=1tMBkjT_RQC4hPl0hCj9reNUjeM4wiIci&export=download',
        'expected_row_cnt': 188
    },
]


def download(source_url, local_temp_loc):
    urllib.request.urlretrieve(source_url,local_temp_loc)


def drop_table_if_exists(table):
    spark.sql(f"drop table if exists {schema}.{table}")


def create_table(table, local_temp_loc):
    spark.read.parquet(local_temp_loc).write.saveAsTable(table)


def validate(table, expected_cnt):
    assert spark.table(table).count() == expected_cnt


def load_data():
    for cfg in config:
        table = cfg['table']
        expected_row_cnt = cfg['expected_row_cnt']
        url = cfg['url']
        local_temp = f'/home/jovyan/data/{table}.snappy.parquet'

        # download(url, local_temp)
        drop_table_if_exists(table)
        create_table(table, f'file:/opt/spark/warehouse/{table}.snappy.parquet')
        validate(table, expected_row_cnt)


load_data()


26/03/20 17:23:54 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/03/20 17:23:54 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/03/20 17:23:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/20 17:23:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [27]:
!ls /home/jovyan/data/

uam_categories.snappy.parquet


In [8]:
!ls /opt/spark/warehouse/

uam_categories.snappy.parquet  uam_orders.snappy.parquet
uam_offers.snappy.parquet
